# 05E · One-time locked independent analysis

**Run only after all five sealed Stage-05D shard ZIPs are present**

This notebook was fixed before independent inference. It first verifies every byte and
every cross-shard lock while results remain sealed. Only after all five archives pass does
it perform the one-time unsealing, frozen H1/H2 tests, 10,000-replicate source bootstrap,
100,000 sign flips per hypothesis, Holm correction, calibration assessment, secondary
summaries, and diagnostic figures.

No GPU is required. Do not edit the implementation or analysis lock.


### 1. Mount Drive and load the precommitted analysis

The implementation and analysis lock are embedded byte-for-byte and hash-checked. The
lock records the exact Stage-05D runner and transition hashes, all statistical conventions,
and the fixed five-shard source map.


In [ ]:
import hashlib
import json
import types
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image as DisplayImage, display

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = Path('/content/drive/MyDrive/reliable-reconstruction-under-mismatch')
else:
    PROJECT_DIR = Path.cwd()
    if PROJECT_DIR.name == 'notebooks':
        PROJECT_DIR = PROJECT_DIR.parent
print('Project:', PROJECT_DIR)


In [ ]:
#@title 🔒 Embedded Stage 05E implementation and analysis lock (expand only for audit) { display-mode: "form" }
ANALYSIS_SOURCE = '"""Pre-locked one-time unsealing and analysis for Experiment 05.\n\nThe implementation is committed before any Stage-05D result exists.  It accepts\nexactly the five fixed sealed shards, verifies every manifested byte and lock,\nthen performs the frozen H1/H2 tests, source bootstrap, Holm correction,\ncalibration assessment, secondary summaries, and diagnostic figures.\n"""\n\nfrom __future__ import annotations\n\nimport hashlib\nimport io\nimport json\nimport math\nimport uuid\nimport zipfile\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\n\nimport numpy as np\nimport pandas as pd\n\n\nEXPERIMENT_ID = "independent_05"\nSTAGE = "05E_one_time_locked_analysis"\nPROTOCOL_SHA256 = "b92c6cf73e05f60dc1edb3a31d88d91623e9ae0cf63d6265e6398e335928794c"\nDATA_RECEIPT_SHA256 = "ad48eb7a65b043173f1012035b4a4c95d0299420614311bdfc6611ef840169cb"\nRUN_CODE_SHA256 = "3fb930119c0e386051765159970234a92a20455d1c3b91615db3c4dddbc87727"\nTRANSITION_SHA256 = "b32ec998761aa4ba4ebe06503aaca3ebe5bba4ddc1c1ec94be4c7a1f0fa84ae2"\nSHARD_COUNT = 5\nSOURCES_PER_SHARD = 8\nCHAIN_IDS = (\n    "q8_b16_n2",\n    "j90_b16_n2",\n    "j75_b16_n2",\n    "j50_b16_n2",\n    "j75_b12_n2",\n    "j75_b20_n2",\n    "j75_b16_n5",\n)\nPIPELINES = ("dpir_nominal", "fbcnn_dpir_nominal")\nHEURISTIC_SCORES = (\n    "operator_spread_detail",\n    "operator_spread_rgb",\n    "image_transform_spread_detail",\n    "original_measurement_residual",\n    "reconstruction_gradient",\n)\nENSEMBLE_SCORE = "fbcnn_dpir_nominal__trained_image_only_patcherrornet_ensemble"\nENSEMBLE_VARIANCE = "fbcnn_dpir_nominal__patcherrornet_ensemble_variance"\nOPERATIONAL_SCORE_KEYS = tuple(\n    f"{pipeline}__score__{score}"\n    for pipeline in PIPELINES\n    for score in HEURISTIC_SCORES\n) + (ENSEMBLE_SCORE,)\nBOOTSTRAP_SEED = 20260921\nBOOTSTRAP_REPLICATES = 10_000\nSIGN_FLIP_SEED = 20260921\nSIGN_FLIP_REPLICATES = 100_000\nPRIMARY_CHAIN = "j75_b16_n2"\nPRIMARY_COVERAGE = 0.5\nPRACTICAL_REDUCTION = 0.05\nPATCHES_PER_OBSERVATION = 1024\nCALIBRATION_BINS = 10\n\n\ndef sha256_bytes(payload: bytes) -> str:\n    return hashlib.sha256(payload).hexdigest()\n\n\ndef sha256_file(path: Path) -> str:\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef _normalize(value):\n    if isinstance(value, dict):\n        return {str(key): _normalize(item) for key, item in value.items()}\n    if isinstance(value, (list, tuple)):\n        return [_normalize(item) for item in value]\n    if isinstance(value, np.generic):\n        return value.item()\n    return value\n\n\ndef dump_json(path: Path, value) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + f".{uuid.uuid4().hex}.partial")\n    temporary.write_text(\n        json.dumps(_normalize(value), indent=2, sort_keys=True, allow_nan=False) + "\\n",\n        encoding="utf-8",\n    )\n    temporary.replace(path)\n\n\ndef dump_csv(path: Path, frame: pd.DataFrame) -> None:\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_suffix(path.suffix + f".{uuid.uuid4().hex}.partial")\n    frame.to_csv(temporary, index=False)\n    temporary.replace(path)\n\n\ndef _archive_prefix(names: list[str]) -> str:\n    assert names and len(names) == len(set(names)), "duplicate ZIP entries"\n    assert not any(\n        name.startswith("/") or ".." in PurePosixPath(name).parts for name in names\n    ), "unsafe ZIP path"\n    if "export_manifest.json" in names:\n        return ""\n    candidates = [name for name in names if name.endswith("/export_manifest.json")]\n    assert len(candidates) == 1, "missing or ambiguous export manifest"\n    prefix = candidates[0][: -len("export_manifest.json")]\n    assert prefix.count("/") == 1\n    return prefix\n\n\ndef validate_analysis_lock(lock_text: str, analysis_code_sha256: str) -> dict:\n    lock = json.loads(lock_text)\n    assert lock["experiment_id"] == EXPERIMENT_ID\n    assert lock["stage"] == STAGE\n    assert lock["outcome_blind"] is True\n    assert lock["independent_results_existed_when_locked"] is False\n    assert lock["analysis_code_sha256"] == analysis_code_sha256\n    assert lock["protocol_sha256"] == PROTOCOL_SHA256\n    assert lock["data_receipt_sha256"] == DATA_RECEIPT_SHA256\n    assert lock["stage_05d_run_code_sha256"] == RUN_CODE_SHA256\n    assert lock["stage_05d_transition_sha256"] == TRANSITION_SHA256\n    assert lock["bootstrap"]["replicates"] == BOOTSTRAP_REPLICATES\n    assert lock["bootstrap"]["seed"] == BOOTSTRAP_SEED\n    assert lock["sign_flip"]["replicates"] == SIGN_FLIP_REPLICATES\n    assert lock["sign_flip"]["seed"] == SIGN_FLIP_SEED\n    assert lock["h2_region"] == "all"\n    assert lock["calibration"]["bins"] == CALIBRATION_BINS\n    return lock\n\n\ndef shard_identity(path: Path) -> int:\n    with zipfile.ZipFile(path) as archive:\n        prefix = _archive_prefix(archive.namelist())\n        manifest = json.loads(archive.read(prefix + "export_manifest.json"))\n    assert manifest["stage"] == "05D_locked_independent_inference"\n    return int(manifest["shard_index"])\n\n\ndef _expected_shard_files(source_ids: list[str]) -> set[str]:\n    observation_ids = [f"{source}_{chain}" for source in source_ids for chain in CHAIN_IDS]\n    result = {\n        "config.json",\n        "lock.json",\n        "provenance.json",\n        "source_manifest.csv",\n        "acquisition_repeat_check.csv",\n        "acquisition.csv",\n        "quality.csv",\n        "risk.csv",\n        "compute.csv",\n        "trajectories.csv",\n        "fbcnn_diagnostics.csv",\n        "checks.json",\n        "status.json",\n    }\n    result.update(f"compact/{observation_id}.npz" for observation_id in observation_ids)\n    result.update(f"records/{observation_id}.json" for observation_id in observation_ids)\n    assert len(result) == 125\n    return result\n\n\ndef inspect_shard(path: Path, analysis_lock: dict) -> dict:\n    """Verify one sealed shard completely before returning any outcome payload."""\n\n    path = Path(path)\n    assert path.is_file()\n    with zipfile.ZipFile(path) as archive:\n        assert archive.testzip() is None\n        names = archive.namelist()\n        prefix = _archive_prefix(names)\n\n        def read(relative: str) -> bytes:\n            return archive.read(prefix + relative)\n\n        manifest_payload = read("export_manifest.json")\n        manifest = json.loads(manifest_payload)\n        index = int(manifest["shard_index"])\n        assert 0 <= index < SHARD_COUNT\n        expected_sources = analysis_lock["source_ids_by_shard"][str(index)]\n        assert manifest["experiment_id"] == EXPERIMENT_ID\n        assert manifest["stage"] == "05D_locked_independent_inference"\n        assert manifest["role"] == "sealed_independent_test"\n        assert manifest["status"] == "passed_locked_independent_shard"\n        assert manifest["shard_count"] == SHARD_COUNT\n        assert manifest["source_ids"] == expected_sources\n        assert tuple(manifest["chain_ids"]) == CHAIN_IDS\n        assert manifest["independent_test_run_authorized"] is True\n        assert manifest["test_inference_performed"] is True\n        assert manifest["test_performance_inspected"] is False\n        assert manifest["aggregate_analysis_performed"] is False\n        assert manifest["unsealed"] is False\n        expected_lock = {\n            "protocol_sha256": PROTOCOL_SHA256,\n            "data_receipt_sha256": DATA_RECEIPT_SHA256,\n            "transition_sha256": TRANSITION_SHA256,\n            "run_code_sha256": RUN_CODE_SHA256,\n            "test_archive_sha256": analysis_lock["test_archive_sha256"],\n            "reliability_archive_sha256": analysis_lock["reliability_archive_sha256"],\n        }\n        assert manifest["lock"] == expected_lock\n        listed = {row["path"]: row for row in manifest["files"]}\n        assert len(listed) == len(manifest["files"])\n        assert set(listed) == _expected_shard_files(expected_sources)\n        actual = {\n            name[len(prefix) :]\n            for name in names\n            if not name.endswith("/") and name != prefix + "export_manifest.json"\n        }\n        assert actual == set(listed)\n        for relative, row in listed.items():\n            payload = read(relative)\n            assert len(payload) == int(row["byte_count"]), relative\n            assert sha256_bytes(payload) == row["sha256"], relative\n\n        config = json.loads(read("config.json"))\n        lock = json.loads(read("lock.json"))\n        status = json.loads(read("status.json"))\n        checks = json.loads(read("checks.json"))\n        assert config["source_ids"] == expected_sources\n        assert config["shard_index"] == index and config["shard_count"] == SHARD_COUNT\n        assert config["test_inference_authorized"] is True\n        assert config["test_performance_inspection_authorized"] is False\n        assert lock == expected_lock\n        assert status["status"] == "passed_locked_independent_shard"\n        assert status["completed_observations"] == status["expected_observations"] == 56\n        assert status["test_performance_inspected"] is False and status["unsealed"] is False\n        assert checks["observations"] == 56 and checks["compact_bundles"] == 56\n        assert checks["ensemble_members"] == 5 and checks["calibration_mappings"] == 11\n        assert checks["test_performance_inspected"] is False and checks["unsealed"] is False\n\n    return {\n        "shard_index": index,\n        "path": str(path),\n        "archive_filename": path.name,\n        "archive_byte_count": path.stat().st_size,\n        "archive_sha256": sha256_file(path),\n        "archive_prefix": prefix,\n        "export_manifest_sha256": sha256_bytes(manifest_payload),\n        "manifest_files_checked": len(listed),\n        "source_ids": expected_sources,\n        "status": "pass_sealed_integrity",\n    }\n\n\ndef _read_verified_payload(path: Path, prefix: str, relative: str) -> bytes:\n    with zipfile.ZipFile(path) as archive:\n        return archive.read(prefix + relative)\n\n\ndef bootstrap_multiplicities(source_count: int) -> np.ndarray:\n    assert source_count == 40\n    rng = np.random.default_rng(BOOTSTRAP_SEED)\n    result = rng.multinomial(\n        source_count,\n        np.full(source_count, 1.0 / source_count),\n        size=BOOTSTRAP_REPLICATES,\n    ).astype(np.int16)\n    assert result.shape == (BOOTSTRAP_REPLICATES, source_count)\n    assert np.all(result.sum(axis=1) == source_count)\n    return result\n\n\ndef percentile_interval(replicates: np.ndarray) -> tuple[float, float]:\n    values = np.asarray(replicates, dtype=np.float64)\n    values = values[np.isfinite(values)]\n    assert len(values) >= int(0.99 * BOOTSTRAP_REPLICATES)\n    low, high = np.quantile(values, [0.025, 0.975], method="linear")\n    return float(low), float(high)\n\n\ndef source_mean_bootstrap(values: np.ndarray, multiplicities: np.ndarray) -> np.ndarray:\n    values = np.asarray(values, dtype=np.float64)\n    assert values.shape == (multiplicities.shape[1],) and np.isfinite(values).all()\n    return multiplicities @ values / multiplicities.shape[1]\n\n\ndef one_sided_sign_flip_pvalue(differences: np.ndarray) -> dict:\n    differences = np.asarray(differences, dtype=np.float64)\n    assert differences.ndim == 1 and np.isfinite(differences).all()\n    observed = float(differences.mean())\n    rng = np.random.default_rng(SIGN_FLIP_SEED)\n    extreme = 0\n    generated = 0\n    batch_size = 5_000\n    while generated < SIGN_FLIP_REPLICATES:\n        count = min(batch_size, SIGN_FLIP_REPLICATES - generated)\n        signs = rng.integers(0, 2, size=(count, len(differences)), dtype=np.int8)\n        signs = signs.astype(np.float64) * 2.0 - 1.0\n        randomized = signs @ differences / len(differences)\n        extreme += int(np.count_nonzero(randomized <= observed))\n        generated += count\n    p_value = (extreme + 1.0) / (SIGN_FLIP_REPLICATES + 1.0)\n    return {\n        "observed_mean_difference": observed,\n        "extreme_randomizations": extreme,\n        "randomizations": SIGN_FLIP_REPLICATES,\n        "p_value_one_sided": float(p_value),\n        "continuity_correction": "(extreme + 1) / (randomizations + 1)",\n    }\n\n\ndef holm_adjust(p_values: dict[str, float]) -> dict[str, float]:\n    ordered = sorted(p_values, key=lambda key: (p_values[key], key))\n    adjusted, running = {}, 0.0\n    total = len(ordered)\n    for rank, key in enumerate(ordered):\n        running = max(running, min(1.0, (total - rank) * float(p_values[key])))\n        adjusted[key] = running\n    return adjusted\n\n\ndef primary_analysis(\n    quality: pd.DataFrame,\n    risk: pd.DataFrame,\n    source_ids: list[str],\n    multiplicities: np.ndarray,\n) -> tuple[dict, pd.DataFrame]:\n    h1_rows = quality[\n        (quality.chain_id == PRIMARY_CHAIN)\n        & quality.method.isin(["dpir_nominal", "fbcnn_dpir_nominal"])\n    ]\n    h1 = h1_rows.pivot(index="source_id", columns="method", values="detail_mse").reindex(\n        source_ids\n    )\n    assert h1.notna().all().all() and h1.shape == (40, 2)\n    h1_difference = (\n        h1["fbcnn_dpir_nominal"] - h1["dpir_nominal"]\n    ).to_numpy(dtype=np.float64)\n\n    h2_rows = risk[\n        (risk.chain_id == PRIMARY_CHAIN)\n        & (risk.pipeline == "fbcnn_dpir_nominal")\n        & (risk.region == "all")\n        & np.isclose(risk.coverage.astype(float), PRIMARY_COVERAGE)\n        & risk.score.isin(["operator_spread_detail", "image_transform_spread_detail"])\n    ]\n    h2 = h2_rows.pivot(index="source_id", columns="score", values="detail_mse").reindex(\n        source_ids\n    )\n    assert h2.notna().all().all() and h2.shape == (40, 2)\n    h2_difference = (\n        h2["operator_spread_detail"] - h2["image_transform_spread_detail"]\n    ).to_numpy(dtype=np.float64)\n\n    contrasts = pd.DataFrame(\n        {\n            "source_id": source_ids,\n            "h1_dpir_detail_mse": h1["dpir_nominal"].to_numpy(),\n            "h1_fbcnn_dpir_detail_mse": h1["fbcnn_dpir_nominal"].to_numpy(),\n            "h1_difference": h1_difference,\n            "h2_image_transform_detail_risk": h2[\n                "image_transform_spread_detail"\n            ].to_numpy(),\n            "h2_operator_spread_detail_risk": h2["operator_spread_detail"].to_numpy(),\n            "h2_difference": h2_difference,\n        }\n    )\n\n    definitions = {\n        "H1_reconstruction": {\n            "difference": h1_difference,\n            "comparator": h1["dpir_nominal"].to_numpy(dtype=np.float64),\n            "contrast": "fbcnn_dpir_nominal minus dpir_nominal",\n            "endpoint": "source detail MSE at j75_b16_n2",\n        },\n        "H2_selection": {\n            "difference": h2_difference,\n            "comparator": h2["image_transform_spread_detail"].to_numpy(dtype=np.float64),\n            "contrast": "operator_spread_detail risk minus image_transform_spread_detail risk",\n            "endpoint": "source retained-patch detail MSE at j75_b16_n2, coverage 0.5, region all",\n        },\n    }\n    results = {}\n    raw_p = {}\n    for hypothesis, definition in definitions.items():\n        differences = definition["difference"]\n        bootstrap = source_mean_bootstrap(differences, multiplicities)\n        ci_low, ci_high = percentile_interval(bootstrap)\n        sign_flip = one_sided_sign_flip_pvalue(differences)\n        comparator_mean = float(np.mean(definition["comparator"]))\n        favorable_reduction = float(-differences.mean() / comparator_mean)\n        raw_p[hypothesis] = sign_flip["p_value_one_sided"]\n        results[hypothesis] = {\n            "unit": "source",\n            "sources": len(source_ids),\n            "contrast": definition["contrast"],\n            "endpoint": definition["endpoint"],\n            "mean_difference": float(differences.mean()),\n            "paired_source_bootstrap_95_ci": [ci_low, ci_high],\n            "comparator_mean": comparator_mean,\n            "relative_reduction": favorable_reduction,\n            "practical_gate_threshold": PRACTICAL_REDUCTION,\n            "practical_gate_passed": favorable_reduction >= PRACTICAL_REDUCTION,\n            **sign_flip,\n        }\n    adjusted = holm_adjust(raw_p)\n    for hypothesis, result in results.items():\n        result["holm_adjusted_p_value"] = adjusted[hypothesis]\n        result["statistical_gate_passed"] = bool(\n            adjusted[hypothesis] < 0.05\n            and result["paired_source_bootstrap_95_ci"][1] < 0.0\n        )\n        result["confirmatory_gate_passed"] = bool(\n            result["practical_gate_passed"] and result["statistical_gate_passed"]\n        )\n    overall = bool(all(row["confirmatory_gate_passed"] for row in results.values()))\n    return {\n        "experiment_id": EXPERIMENT_ID,\n        "stage": STAGE,\n        "multiplicity": "Holm across H1 and H2",\n        "hypotheses": results,\n        "both_confirmatory_gates_passed": overall,\n        "experimental_novelty_gate_passed": overall,\n        "final_literature_novelty_claim_established": False,\n        "claim_boundary": (\n            "Passing both gates establishes the predeclared independent experimental gate only. "\n            "The final novelty claim still requires Stage-05F synthesis against the locked literature boundary."\n        ),\n    }, contrasts\n\n\ndef _equal_mass_assignments(probability: np.ndarray, weights: np.ndarray) -> np.ndarray:\n    order = np.argsort(probability, kind="mergesort")\n    cumulative = np.cumsum(weights[order]) - 0.5 * weights[order]\n    ordered_bins = np.minimum(\n        (cumulative / weights.sum() * CALIBRATION_BINS).astype(np.int8),\n        CALIBRATION_BINS - 1,\n    )\n    assignments = np.empty(len(order), dtype=np.int8)\n    assignments[order] = ordered_bins\n    return assignments\n\n\ndef _logistic_slopes(\n    source_mass: np.ndarray,\n    source_positive: np.ndarray,\n    logit_probability: np.ndarray,\n    multiplicities: np.ndarray,\n) -> np.ndarray:\n    mass = multiplicities @ source_mass\n    positive = multiplicities @ source_positive\n    beta = np.zeros((len(multiplicities), 2), dtype=np.float64)\n    x = logit_probability[None, :]\n    for _ in range(30):\n        eta = np.clip(beta[:, :1] + beta[:, 1:] * x, -30.0, 30.0)\n        mu = 1.0 / (1.0 + np.exp(-eta))\n        residual = positive - mass * mu\n        variance = mass * mu * (1.0 - mu)\n        g0 = residual.sum(axis=1)\n        g1 = (residual * x).sum(axis=1)\n        h00 = variance.sum(axis=1)\n        h01 = (variance * x).sum(axis=1)\n        h11 = (variance * x * x).sum(axis=1)\n        determinant = h00 * h11 - h01 * h01\n        valid = determinant > 1e-14\n        delta0 = np.zeros(len(beta), dtype=np.float64)\n        delta1 = np.zeros(len(beta), dtype=np.float64)\n        delta0[valid] = (h11[valid] * g0[valid] - h01[valid] * g1[valid]) / determinant[valid]\n        delta1[valid] = (-h01[valid] * g0[valid] + h00[valid] * g1[valid]) / determinant[valid]\n        beta[:, 0] += delta0\n        beta[:, 1] += delta1\n        if float(np.max(np.abs(np.column_stack([delta0, delta1])))) < 1e-9:\n            break\n    beta[~np.isfinite(beta)] = np.nan\n    return beta[:, 1]\n\n\ndef calibration_assessment(\n    probabilities: dict[str, np.ndarray],\n    target: np.ndarray,\n    source_index: np.ndarray,\n    chain_index: np.ndarray,\n    source_ids: list[str],\n    multiplicities: np.ndarray,\n) -> tuple[pd.DataFrame, pd.DataFrame]:\n    target = np.asarray(target, dtype=np.float64)\n    assert len(target) == 40 * len(CHAIN_IDS) * PATCHES_PER_OBSERVATION\n    source_counts = np.bincount(source_index, minlength=40)\n    assert np.all(source_counts == len(CHAIN_IDS) * PATCHES_PER_OBSERVATION)\n    weights = 1.0 / source_counts[source_index]\n    metric_rows, bin_rows = [], []\n\n    for score_key in OPERATIONAL_SCORE_KEYS:\n        probability = np.asarray(probabilities[score_key], dtype=np.float64)\n        assert probability.shape == target.shape\n        assert np.isfinite(probability).all() and 0 <= probability.min() <= probability.max() <= 1\n        assignments = _equal_mass_assignments(probability, weights)\n        source_brier = np.zeros(40, dtype=np.float64)\n        source_citl = np.zeros(40, dtype=np.float64)\n        source_bin_mass = np.zeros((40, CALIBRATION_BINS), dtype=np.float64)\n        source_bin_probability = np.zeros_like(source_bin_mass)\n        source_bin_target = np.zeros_like(source_bin_mass)\n        for source in range(40):\n            selected_source = source_index == source\n            source_brier[source] = np.mean((probability[selected_source] - target[selected_source]) ** 2)\n            source_citl[source] = np.mean(target[selected_source] - probability[selected_source])\n            for bin_index in range(CALIBRATION_BINS):\n                selected = selected_source & (assignments == bin_index)\n                source_bin_mass[source, bin_index] = weights[selected].sum()\n                source_bin_probability[source, bin_index] = np.sum(weights[selected] * probability[selected])\n                source_bin_target[source, bin_index] = np.sum(weights[selected] * target[selected])\n\n        point_mass = source_bin_mass.sum(axis=0)\n        point_probability = source_bin_probability.sum(axis=0)\n        point_target = source_bin_target.sum(axis=0)\n        ece = float(np.abs(point_target - point_probability).sum() / 40.0)\n        for bin_index in range(CALIBRATION_BINS):\n            mass = point_mass[bin_index]\n            bin_rows.append(\n                {\n                    "scope": "all_chains",\n                    "score_key": score_key,\n                    "bin": bin_index,\n                    "weight_mass": float(mass),\n                    "mean_probability": float(point_probability[bin_index] / mass),\n                    "observed_bad_detail_rate": float(point_target[bin_index] / mass),\n                }\n            )\n\n        unique_probability, inverse = np.unique(np.clip(probability, 1e-6, 1 - 1e-6), return_inverse=True)\n        source_unique_mass = np.zeros((40, len(unique_probability)), dtype=np.float64)\n        source_unique_positive = np.zeros_like(source_unique_mass)\n        for source in range(40):\n            selected = source_index == source\n            np.add.at(source_unique_mass[source], inverse[selected], weights[selected])\n            np.add.at(\n                source_unique_positive[source],\n                inverse[selected],\n                weights[selected] * target[selected],\n            )\n        logit = np.log(unique_probability / (1.0 - unique_probability))\n        point_multiplicity = np.ones((1, 40), dtype=np.int16)\n        slope = float(\n            _logistic_slopes(\n                source_unique_mass, source_unique_positive, logit, point_multiplicity\n            )[0]\n        )\n\n        bootstrap_brier = multiplicities @ source_brier / 40.0\n        bootstrap_citl = multiplicities @ source_citl / 40.0\n        bootstrap_probability = multiplicities @ source_bin_probability\n        bootstrap_target = multiplicities @ source_bin_target\n        bootstrap_ece = np.abs(bootstrap_target - bootstrap_probability).sum(axis=1) / 40.0\n        bootstrap_slope = _logistic_slopes(\n            source_unique_mass, source_unique_positive, logit, multiplicities\n        )\n        metrics = {\n            "source_macro_brier": (float(source_brier.mean()), bootstrap_brier),\n            "equal_mass_ece_10bin": (ece, bootstrap_ece),\n            "calibration_in_the_large": (float(source_citl.mean()), bootstrap_citl),\n            "calibration_slope": (slope, bootstrap_slope),\n        }\n        for metric, (estimate, replicates) in metrics.items():\n            ci_low, ci_high = percentile_interval(replicates)\n            metric_rows.append(\n                {\n                    "scope": "all_chains",\n                    "score_key": score_key,\n                    "metric": metric,\n                    "estimate": estimate,\n                    "bootstrap_ci_low": ci_low,\n                    "bootstrap_ci_high": ci_high,\n                    "sources": 40,\n                    "patches": len(target),\n                }\n            )\n\n        for chain_number, chain_id in enumerate(CHAIN_IDS):\n            selected = chain_index == chain_number\n            local_probability, local_target = probability[selected], target[selected]\n            local_source = source_index[selected]\n            local_counts = np.bincount(local_source, minlength=40)\n            local_weights = 1.0 / local_counts[local_source]\n            local_assignments = _equal_mass_assignments(local_probability, local_weights)\n            local_brier = np.mean(\n                [\n                    np.mean((local_probability[local_source == source] - local_target[local_source == source]) ** 2)\n                    for source in range(40)\n                ]\n            )\n            local_citl = np.mean(\n                [\n                    np.mean(local_target[local_source == source] - local_probability[local_source == source])\n                    for source in range(40)\n                ]\n            )\n            local_ece = 0.0\n            for bin_index in range(CALIBRATION_BINS):\n                in_bin = local_assignments == bin_index\n                local_ece += abs(\n                    float(np.sum(local_weights[in_bin] * (local_target[in_bin] - local_probability[in_bin])))\n                ) / 40.0\n            for metric, estimate in (\n                ("source_macro_brier", local_brier),\n                ("equal_mass_ece_10bin", local_ece),\n                ("calibration_in_the_large", local_citl),\n            ):\n                metric_rows.append(\n                    {\n                        "scope": chain_id,\n                        "score_key": score_key,\n                        "metric": metric,\n                        "estimate": float(estimate),\n                        "bootstrap_ci_low": np.nan,\n                        "bootstrap_ci_high": np.nan,\n                        "sources": 40,\n                        "patches": int(selected.sum()),\n                    }\n                )\n    return pd.DataFrame(metric_rows), pd.DataFrame(bin_rows)\n\n\ndef source_group_summary(\n    frame: pd.DataFrame,\n    group_columns: list[str],\n    metric_columns: list[str],\n    source_ids: list[str],\n    multiplicities: np.ndarray,\n) -> pd.DataFrame:\n    rows = []\n    for keys, group in frame.groupby(group_columns, sort=True, dropna=False):\n        keys = keys if isinstance(keys, tuple) else (keys,)\n        group = group.set_index("source_id").reindex(source_ids)\n        assert group[metric_columns].notna().all().all()\n        row = dict(zip(group_columns, keys))\n        row["sources"] = len(source_ids)\n        for metric in metric_columns:\n            values = group[metric].to_numpy(dtype=np.float64)\n            replicates = source_mean_bootstrap(values, multiplicities)\n            low, high = percentile_interval(replicates)\n            row[f"mean_{metric}"] = float(values.mean())\n            row[f"{metric}_bootstrap_ci_low"] = low\n            row[f"{metric}_bootstrap_ci_high"] = high\n        rows.append(row)\n    return pd.DataFrame(rows)\n\n\ndef _compact_expected_arrays() -> set[str]:\n    result = {\n        "observation_uint8",\n        "fbcnn_dpir_nominal",\n        "dpir_nominal__rgb_patch_error",\n        "dpir_nominal__detail_patch_error",\n        "fbcnn_dpir_nominal__rgb_patch_error",\n        "fbcnn_dpir_nominal__detail_patch_error",\n        "target_bad_detail_0p05",\n        "reference_texture",\n        ENSEMBLE_VARIANCE,\n    }\n    result.update(OPERATIONAL_SCORE_KEYS)\n    result.update(f"{key}__calibrated_probability" for key in OPERATIONAL_SCORE_KEYS)\n    assert len(result) == 31\n    return result\n\n\ndef load_unsealed_data(verified: list[dict]) -> dict:\n    quality_frames, risk_frames, source_frames = [], [], []\n    probabilities = {key: [] for key in OPERATIONAL_SCORE_KEYS}\n    targets, sources, chains = [], [], []\n    all_source_ids = sorted(source for report in verified for source in report["source_ids"])\n    assert len(all_source_ids) == len(set(all_source_ids)) == 40\n    source_number = {source: index for index, source in enumerate(all_source_ids)}\n\n    for report in sorted(verified, key=lambda row: row["shard_index"]):\n        path, prefix = Path(report["path"]), report["archive_prefix"]\n        quality_frames.append(\n            pd.read_csv(io.BytesIO(_read_verified_payload(path, prefix, "quality.csv")))\n        )\n        risk_frames.append(pd.read_csv(io.BytesIO(_read_verified_payload(path, prefix, "risk.csv"))))\n        source_frames.append(\n            pd.read_csv(io.BytesIO(_read_verified_payload(path, prefix, "source_manifest.csv")))\n        )\n        with zipfile.ZipFile(path) as archive:\n            for source_id in report["source_ids"]:\n                for chain_index, chain_id in enumerate(CHAIN_IDS):\n                    relative = f"compact/{source_id}_{chain_id}.npz"\n                    payload = archive.read(prefix + relative)\n                    with np.load(io.BytesIO(payload), allow_pickle=False) as stored:\n                        assert set(stored.files) == _compact_expected_arrays()\n                        target = stored["target_bad_detail_0p05"].astype(np.uint8)\n                        assert target.shape == (PATCHES_PER_OBSERVATION,)\n                        assert set(np.unique(target)).issubset({0, 1})\n                        targets.append(target)\n                        sources.append(\n                            np.full(PATCHES_PER_OBSERVATION, source_number[source_id], dtype=np.uint8)\n                        )\n                        chains.append(\n                            np.full(PATCHES_PER_OBSERVATION, chain_index, dtype=np.uint8)\n                        )\n                        for score_key in OPERATIONAL_SCORE_KEYS:\n                            values = stored[f"{score_key}__calibrated_probability"].astype(\n                                np.float32\n                            )\n                            assert values.shape == (PATCHES_PER_OBSERVATION,)\n                            assert np.isfinite(values).all() and 0 <= values.min() <= values.max() <= 1\n                            probabilities[score_key].append(values)\n\n    quality = pd.concat(quality_frames, ignore_index=True)\n    risk = pd.concat(risk_frames, ignore_index=True)\n    source_manifest = pd.concat(source_frames, ignore_index=True)\n    assert len(source_manifest) == 40 and source_manifest.source_id.nunique() == 40\n    assert len(quality) == 40 * len(CHAIN_IDS) * 6\n    assert not quality.duplicated(["source_id", "chain_id", "method"]).any()\n    assert len(risk) == 40 * len(CHAIN_IDS) * 120\n    assert not risk.duplicated(\n        ["source_id", "chain_id", "pipeline", "region", "score_key", "coverage"]\n    ).any()\n    return {\n        "source_ids": all_source_ids,\n        "source_manifest": source_manifest.sort_values("source_id").reset_index(drop=True),\n        "quality": quality,\n        "risk": risk,\n        "probabilities": {key: np.concatenate(value) for key, value in probabilities.items()},\n        "target": np.concatenate(targets),\n        "source_index": np.concatenate(sources),\n        "chain_index": np.concatenate(chains),\n    }\n\n\ndef build_figures(\n    primary: dict,\n    risk_summary: pd.DataFrame,\n    calibration_bins: pd.DataFrame,\n    output_dir: Path,\n) -> list[str]:\n    import matplotlib.pyplot as plt\n\n    figure_dir = Path(output_dir) / "figures"\n    figure_dir.mkdir(exist_ok=True)\n    plt.rcParams.update(\n        {\n            "font.size": 9,\n            "axes.spines.top": False,\n            "axes.spines.right": False,\n            "axes.grid": True,\n            "grid.alpha": 0.2,\n            "figure.facecolor": "white",\n            "savefig.facecolor": "white",\n        }\n    )\n    paths = []\n\n    names = ["H1 reconstruction", "H2 selection"]\n    records = [primary["hypotheses"]["H1_reconstruction"], primary["hypotheses"]["H2_selection"]]\n    estimates = [row["mean_difference"] for row in records]\n    low = [row["paired_source_bootstrap_95_ci"][0] for row in records]\n    high = [row["paired_source_bootstrap_95_ci"][1] for row in records]\n    fig, axis = plt.subplots(figsize=(7.2, 3.8))\n    y = np.arange(2)\n    axis.errorbar(\n        estimates,\n        y,\n        xerr=[np.asarray(estimates) - np.asarray(low), np.asarray(high) - np.asarray(estimates)],\n        fmt="o",\n        color="#1f4e79",\n        capsize=4,\n    )\n    axis.axvline(0, color="#444444", linestyle="--", linewidth=1)\n    axis.set_yticks(y, names)\n    axis.invert_yaxis()\n    axis.set_xlabel("Paired source-mean difference (negative is favorable)")\n    axis.set_title("Independent confirmatory contrasts with 95% source-bootstrap intervals")\n    fig.tight_layout()\n    path = figure_dir / "primary_contrasts.png"\n    fig.savefig(path, dpi=180)\n    plt.close(fig)\n    paths.append(str(path))\n\n    selected = risk_summary[\n        (risk_summary.chain_id == PRIMARY_CHAIN)\n        & (risk_summary.pipeline == "fbcnn_dpir_nominal")\n        & (risk_summary.region == "all")\n        & risk_summary.score.isin(\n            [\n                "operator_spread_detail",\n                "image_transform_spread_detail",\n                "trained_image_only_patcherrornet_ensemble",\n                "expected_random",\n                "oracle_detail_error",\n            ]\n        )\n    ]\n    fig, axis = plt.subplots(figsize=(7.2, 4.5))\n    for score, frame in selected.groupby("score"):\n        frame = frame.sort_values("coverage")\n        axis.plot(100 * frame.coverage, frame.mean_detail_mse, marker="o", label=score)\n    axis.set_xlabel("Retained coverage (%)")\n    axis.set_ylabel("Source-mean retained detail MSE")\n    axis.set_title("Primary-chain independent risk–coverage")\n    axis.legend(fontsize=7)\n    fig.tight_layout()\n    path = figure_dir / "primary_risk_coverage.png"\n    fig.savefig(path, dpi=180)\n    plt.close(fig)\n    paths.append(str(path))\n\n    fig, axes = plt.subplots(3, 4, figsize=(13, 9), sharex=True, sharey=True)\n    for axis, score_key in zip(axes.ravel(), OPERATIONAL_SCORE_KEYS):\n        frame = calibration_bins[calibration_bins.score_key == score_key].sort_values("bin")\n        axis.plot([0, 1], [0, 1], color="#555555", linestyle="--", linewidth=0.8)\n        axis.plot(\n            frame.mean_probability,\n            frame.observed_bad_detail_rate,\n            marker="o",\n            color="#1f4e79",\n        )\n        axis.set_title(score_key.replace("__score__", " · ").replace("_", " "), fontsize=7)\n    axes.ravel()[-1].axis("off")\n    fig.supxlabel("Mapped probability")\n    fig.supylabel("Observed bad-detail rate")\n    fig.suptitle("Independent reliability diagrams · all acquisition chains")\n    fig.tight_layout(rect=(0, 0, 1, 0.97))\n    path = figure_dir / "independent_calibration_reliability.png"\n    fig.savefig(path, dpi=180)\n    plt.close(fig)\n    paths.append(str(path))\n    return paths\n\n\ndef run_analysis(archive_paths, output_dir, lock_text: str, analysis_code_sha256: str) -> dict:\n    analysis_lock = validate_analysis_lock(lock_text, analysis_code_sha256)\n    paths = [Path(path) for path in archive_paths]\n    assert len(paths) == SHARD_COUNT\n    by_index = {}\n    for path in paths:\n        index = shard_identity(path)\n        assert index not in by_index, f"duplicate shard index: {index}"\n        by_index[index] = path\n    assert sorted(by_index) == list(range(SHARD_COUNT))\n\n    verified = [inspect_shard(by_index[index], analysis_lock) for index in range(SHARD_COUNT)]\n    all_sources = [source for report in verified for source in report["source_ids"]]\n    assert len(all_sources) == len(set(all_sources)) == 40\n\n    final_output_dir = Path(output_dir)\n    assert not final_output_dir.exists(), (\n        f"one-time unsealing output already exists: {final_output_dir}"\n    )\n    final_output_dir.parent.mkdir(parents=True, exist_ok=True)\n    output_dir = final_output_dir.with_name(\n        final_output_dir.name + f".building.{uuid.uuid4().hex}"\n    )\n    output_dir.mkdir()\n    dump_json(\n        output_dir / "unsealing_receipt.json",\n        {\n            "experiment_id": EXPERIMENT_ID,\n            "stage": STAGE,\n            "unsealed_at_utc": datetime.now(timezone.utc).isoformat(),\n            "analysis_lock_sha256": sha256_bytes(lock_text.encode()),\n            "analysis_code_sha256": analysis_code_sha256,\n            "shards": verified,\n            "sealed_shards_verified_before_unsealing": True,\n            "test_performance_inspected_before_unsealing": False,\n            "unsealed": True,\n        },\n    )\n\n    data = load_unsealed_data(verified)\n    multiplicities = bootstrap_multiplicities(len(data["source_ids"]))\n    primary, contrasts = primary_analysis(\n        data["quality"], data["risk"], data["source_ids"], multiplicities\n    )\n    calibration_metrics, calibration_bins = calibration_assessment(\n        data["probabilities"],\n        data["target"],\n        data["source_index"],\n        data["chain_index"],\n        data["source_ids"],\n        multiplicities,\n    )\n    quality_metrics = [\n        "mse",\n        "psnr_db",\n        "detail_mse",\n        "bad_detail_rate_0.025",\n        "bad_detail_rate_0.05",\n        "bad_detail_rate_0.1",\n    ]\n    risk_metrics = [\n        "rgb_mse",\n        "detail_mse",\n        "bad_detail_rate_0.025",\n        "bad_detail_rate_0.05",\n        "bad_detail_rate_0.1",\n    ]\n    quality_summary = source_group_summary(\n        data["quality"], ["chain_id", "method"], quality_metrics, data["source_ids"], multiplicities\n    )\n    risk_summary = source_group_summary(\n        data["risk"],\n        ["chain_id", "pipeline", "region", "score", "score_key", "score_role", "coverage"],\n        risk_metrics,\n        data["source_ids"],\n        multiplicities,\n    )\n\n    dump_csv(output_dir / "source_manifest.csv", data["source_manifest"])\n    dump_csv(output_dir / "primary_source_contrasts.csv", contrasts)\n    dump_json(output_dir / "primary_hypotheses.json", primary)\n    dump_csv(output_dir / "quality_summary.csv", quality_summary)\n    dump_csv(output_dir / "risk_summary.csv", risk_summary)\n    dump_csv(output_dir / "calibration_metrics.csv", calibration_metrics)\n    dump_csv(output_dir / "calibration_bins.csv", calibration_bins)\n    figures = build_figures(primary, risk_summary, calibration_bins, output_dir)\n\n    checks = {\n        "sealed_shards": 5,\n        "manifest_files_checked": sum(row["manifest_files_checked"] for row in verified),\n        "sources": 40,\n        "chains": 7,\n        "observations": 280,\n        "quality_rows": len(data["quality"]),\n        "risk_rows": len(data["risk"]),\n        "calibration_patch_rows": len(data["target"]),\n        "calibration_scores": len(OPERATIONAL_SCORE_KEYS),\n        "bootstrap_replicates": BOOTSTRAP_REPLICATES,\n        "sign_flip_replicates_per_hypothesis": SIGN_FLIP_REPLICATES,\n        "holm_hypotheses": 2,\n        "figures": [str(Path(path).relative_to(output_dir)) for path in figures],\n        "unsealed": True,\n    }\n    assert checks["quality_rows"] == 1680\n    assert checks["risk_rows"] == 33600\n    assert checks["calibration_patch_rows"] == 286720\n    dump_json(output_dir / "checks.json", checks)\n    dump_json(\n        output_dir / "status.json",\n        {\n            "experiment_id": EXPERIMENT_ID,\n            "stage": STAGE,\n            "status": "passed_one_time_locked_analysis",\n            "independent_test_run_complete": True,\n            "independent_test_performance_inspected": True,\n            "unsealed": True,\n            "both_confirmatory_gates_passed": primary["both_confirmatory_gates_passed"],\n            "experimental_novelty_gate_passed": primary["experimental_novelty_gate_passed"],\n            "final_literature_novelty_claim_established": False,\n            "next_stage": "05F_locked_literature_and_experimental_synthesis",\n            "updated_at_utc": datetime.now(timezone.utc).isoformat(),\n        },\n    )\n\n    manifest_files = []\n    for path in sorted(output_dir.rglob("*")):\n        if path.is_file() and path.name != "export_manifest.json" and ".partial" not in path.name:\n            manifest_files.append(\n                {\n                    "path": str(path.relative_to(output_dir)),\n                    "byte_count": path.stat().st_size,\n                    "sha256": sha256_file(path),\n                }\n            )\n    dump_json(\n        output_dir / "export_manifest.json",\n        {\n            "experiment_id": EXPERIMENT_ID,\n            "stage": STAGE,\n            "status": "passed_one_time_locked_analysis",\n            "unsealed": True,\n            "files": manifest_files,\n        },\n    )\n    output_dir.replace(final_output_dir)\n    output_dir = final_output_dir\n    archive_stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")\n    archive_path = output_dir.parent / f"{output_dir.name}_{archive_stamp}.zip"\n    temporary = archive_path.with_suffix(f".{uuid.uuid4().hex}.partial")\n    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:\n        for row in manifest_files:\n            archive.write(output_dir / row["path"], arcname=row["path"])\n        archive.write(output_dir / "export_manifest.json", arcname="export_manifest.json")\n    temporary.replace(archive_path)\n    return {\n        "status": "passed_one_time_locked_analysis",\n        "output_dir": str(output_dir),\n        "archive_path": str(archive_path),\n        "archive_sha256": sha256_file(archive_path),\n        "both_confirmatory_gates_passed": primary["both_confirmatory_gates_passed"],\n        "experimental_novelty_gate_passed": primary["experimental_novelty_gate_passed"],\n        "final_literature_novelty_claim_established": False,\n        "primary_hypotheses": primary["hypotheses"],\n    }\n\n\ndef static_self_check() -> dict:\n    assert SHARD_COUNT * SOURCES_PER_SHARD == 40\n    assert len(CHAIN_IDS) == 7\n    assert len(OPERATIONAL_SCORE_KEYS) == 11\n    assert BOOTSTRAP_REPLICATES == 10_000 and SIGN_FLIP_REPLICATES == 100_000\n    sample = np.asarray([-2.0, -1.0, 1.0, 2.0])\n    adjusted = holm_adjust({"a": 0.01, "b": 0.04})\n    assert adjusted == {"a": 0.02, "b": 0.04}\n    assert math.isclose(float(sample.mean()), 0.0)\n    return {\n        "experiment_id": EXPERIMENT_ID,\n        "stage": STAGE,\n        "required_shards": SHARD_COUNT,\n        "sources": 40,\n        "operational_scores": len(OPERATIONAL_SCORE_KEYS),\n        "bootstrap_replicates": BOOTSTRAP_REPLICATES,\n        "sign_flip_replicates": SIGN_FLIP_REPLICATES,\n        "primary_hypotheses": 2,\n    }\n'
ANALYSIS_LOCK_TEXT = '{\n  "schema_version": "1.0.0",\n  "experiment_id": "independent_05",\n  "stage": "05E_one_time_locked_analysis",\n  "locked_at_utc": "2026-09-24T23:29:35Z",\n  "author": "Bamidele Akinwumi research workflow (Codex-assisted)",\n  "outcome_blind": true,\n  "independent_results_existed_when_locked": false,\n  "independent_test_inference_performed_when_locked": false,\n  "independent_test_performance_inspected_when_locked": false,\n  "protocol_sha256": "b92c6cf73e05f60dc1edb3a31d88d91623e9ae0cf63d6265e6398e335928794c",\n  "data_receipt_sha256": "ad48eb7a65b043173f1012035b4a4c95d0299420614311bdfc6611ef840169cb",\n  "stage_05d_run_code_sha256": "3fb930119c0e386051765159970234a92a20455d1c3b91615db3c4dddbc87727",\n  "stage_05d_transition_sha256": "b32ec998761aa4ba4ebe06503aaca3ebe5bba4ddc1c1ec94be4c7a1f0fa84ae2",\n  "analysis_code_path": "experiments/independent_05/independent_analysis.py",\n  "analysis_code_sha256": "930ab1dad38ad0c93ba639c32bcf9f626aa9e7d4789d0f206818be06aa7f0154",\n  "test_archive_sha256": "987e6a5208206b51bb556e808ea052aa588f373124bd0f571888a4754c6afa89",\n  "reliability_archive_sha256": "a77f409016070ac799133502c3b24f9603e918b85a3069e539d70fbd8d8cad12",\n  "required_shards": 5,\n  "required_sources": 40,\n  "required_observations": 280,\n  "h1": {\n    "condition": "j75_b16_n2",\n    "contrast": "fbcnn_dpir_nominal minus dpir_nominal",\n    "endpoint": "source detail MSE over the full 512x512 evaluation region",\n    "direction": "negative is favorable",\n    "relative_reduction_formula": "-(mean paired difference) / mean dpir_nominal detail MSE",\n    "practical_threshold": 0.05\n  },\n  "h2": {\n    "condition": "j75_b16_n2",\n    "pipeline": "fbcnn_dpir_nominal",\n    "coverage": 0.5,\n    "region": "all",\n    "contrast": "operator_spread_detail risk minus image_transform_spread_detail risk",\n    "endpoint": "source retained-patch detail MSE",\n    "direction": "negative is favorable",\n    "relative_reduction_formula": "-(mean paired difference) / mean image_transform_spread_detail risk",\n    "practical_threshold": 0.05\n  },\n  "h2_region": "all",\n  "bootstrap": {\n    "unit": "source",\n    "replicates": 10000,\n    "seed": 20260921,\n    "generator": "numpy.random.default_rng(seed).multinomial(40, equal source probabilities)",\n    "interval": "two-sided 95% percentile interval using numpy linear quantiles",\n    "shared_multiplicities": "One outcome-blind multiplicity matrix is reused across primary, secondary and calibration endpoints."\n  },\n  "sign_flip": {\n    "unit": "source paired difference",\n    "alternative": "negative/favorable",\n    "replicates": 100000,\n    "seed": 20260921,\n    "generator": "numpy.random.default_rng(seed), reset separately for H1 and H2",\n    "p_value": "(number of randomized means <= observed mean + 1) / (100000 + 1)"\n  },\n  "multiplicity": {\n    "family": [\n      "H1_reconstruction",\n      "H2_selection"\n    ],\n    "method": "Holm step-down",\n    "alpha": 0.05\n  },\n  "confirmatory_gate": "Each hypothesis must independently pass its 5% practical threshold, Holm-adjusted one-sided p<0.05, and paired-bootstrap upper confidence limit<0.",\n  "calibration": {\n    "scope": "All seven frozen acquisition chains pooled; each of the 40 sources has equal total weight.",\n    "scores": 11,\n    "target": "centre 16x16 patch detail RMSE > 0.05 for fbcnn_dpir_nominal",\n    "bins": 10,\n    "equal_mass_bin_rule": "Stable sort by mapped probability; assign bins from weighted cumulative midpoints on the full independent cohort.",\n    "bootstrap_bin_rule": "Keep full-cohort bin assignments fixed and source-bootstrap the within-bin weighted probability and event totals.",\n    "metrics": [\n      "source-macro Brier score",\n      "10-bin equal-mass expected calibration error",\n      "calibration-in-the-large as source-equal mean(target - mapped probability)",\n      "calibration slope from source-weighted logistic regression of target on logit(mapped probability), clipping probabilities to [1e-6,1-1e-6]"\n    ],\n    "intervals": "10,000-replicate source-bootstrap 95% percentile intervals for all pooled metrics.",\n    "per_chain_outputs": "Point estimates for Brier, ECE and calibration-in-the-large are exploratory diagnostics."\n  },\n  "secondary_outputs": {\n    "quality": "Source-mean estimates and 95% source-bootstrap intervals by chain and reconstruction method.",\n    "risk_coverage": "Source-mean estimates and 95% source-bootstrap intervals by chain, pipeline, region, score and coverage.",\n    "interpretation": "Secondary outcomes cannot rescue a failed confirmatory gate."\n  },\n  "source_ids_by_shard": {\n    "0": [\n      "img_2400x2400_3x8bit_C00C00_RGB_almonds",\n      "img_2400x2400_3x8bit_C00C00_RGB_apples",\n      "img_2400x2400_3x8bit_C00C00_RGB_baloons",\n      "img_2400x2400_3x8bit_C00C00_RGB_bananas",\n      "img_2400x2400_3x8bit_C00C00_RGB_billiard_balls_a",\n      "img_2400x2400_3x8bit_C00C00_RGB_billiard_balls_b",\n      "img_2400x2400_3x8bit_C00C00_RGB_building",\n      "img_2400x2400_3x8bit_C00C00_RGB_cards_a"\n    ],\n    "1": [\n      "img_2400x2400_3x8bit_C00C00_RGB_cards_b",\n      "img_2400x2400_3x8bit_C00C00_RGB_carrots",\n      "img_2400x2400_3x8bit_C00C00_RGB_chairs",\n      "img_2400x2400_3x8bit_C00C00_RGB_clips",\n      "img_2400x2400_3x8bit_C00C00_RGB_coins",\n      "img_2400x2400_3x8bit_C00C00_RGB_cushions",\n      "img_2400x2400_3x8bit_C00C00_RGB_ducks",\n      "img_2400x2400_3x8bit_C00C00_RGB_fence"\n    ],\n    "2": [\n      "img_2400x2400_3x8bit_C00C00_RGB_flowers",\n      "img_2400x2400_3x8bit_C00C00_RGB_garden_table",\n      "img_2400x2400_3x8bit_C00C00_RGB_guitar_bridge",\n      "img_2400x2400_3x8bit_C00C00_RGB_guitar_fret",\n      "img_2400x2400_3x8bit_C00C00_RGB_guitar_head",\n      "img_2400x2400_3x8bit_C00C00_RGB_keyboard_a",\n      "img_2400x2400_3x8bit_C00C00_RGB_keyboard_b",\n      "img_2400x2400_3x8bit_C00C00_RGB_lion"\n    ],\n    "3": [\n      "img_2400x2400_3x8bit_C00C00_RGB_multimeter",\n      "img_2400x2400_3x8bit_C00C00_RGB_pencils_a",\n      "img_2400x2400_3x8bit_C00C00_RGB_pencils_b",\n      "img_2400x2400_3x8bit_C00C00_RGB_pillar",\n      "img_2400x2400_3x8bit_C00C00_RGB_plastic",\n      "img_2400x2400_3x8bit_C00C00_RGB_roof",\n      "img_2400x2400_3x8bit_C00C00_RGB_scarf",\n      "img_2400x2400_3x8bit_C00C00_RGB_screws"\n    ],\n    "4": [\n      "img_2400x2400_3x8bit_C00C00_RGB_snails",\n      "img_2400x2400_3x8bit_C00C00_RGB_socks",\n      "img_2400x2400_3x8bit_C00C00_RGB_sweets",\n      "img_2400x2400_3x8bit_C00C00_RGB_tomatoes_a",\n      "img_2400x2400_3x8bit_C00C00_RGB_tomatoes_b",\n      "img_2400x2400_3x8bit_C00C00_RGB_tools_a",\n      "img_2400x2400_3x8bit_C00C00_RGB_tools_b",\n      "img_2400x2400_3x8bit_C00C00_RGB_wood_game"\n    ]\n  },\n  "one_time_rule": "The analysis refuses an existing final output directory. It verifies all five sealed shards before creating a temporary unsealing workspace and publishes the final directory atomically.",\n  "claim_boundary": "The Stage-05E output decides the predeclared independent experimental gates. It does not by itself establish the final literature novelty claim, which requires the locked Stage-05F synthesis."\n}\n'
EXPECTED_DIGESTS = {'analysis': '930ab1dad38ad0c93ba639c32bcf9f626aa9e7d4789d0f206818be06aa7f0154', 'lock': 'cd216802d0a5eaf8d03e9811d6673021e85c042e4e3e57b29be7cc6f69483987'}

assert hashlib.sha256(ANALYSIS_SOURCE.encode()).hexdigest() == EXPECTED_DIGESTS['analysis']
assert hashlib.sha256(ANALYSIS_LOCK_TEXT.encode()).hexdigest() == EXPECTED_DIGESTS['lock']
IA05 = types.ModuleType('independent_05_locked_analysis_embedded')
exec(compile(ANALYSIS_SOURCE, '<independent_05_locked_analysis_embedded>', 'exec'), IA05.__dict__)
ANALYSIS_LOCK_05E = IA05.validate_analysis_lock(
    ANALYSIS_LOCK_TEXT, EXPECTED_DIGESTS['analysis']
)
print(json.dumps(IA05.static_self_check(), indent=2))
print('Outcome-blind Stage-05E analysis lock: VERIFIED')


### 2. Locate exactly five sealed shard archives

Leave the folder blank to use the project `results/` directory. There must be exactly one
ZIP for each shard index 0–4. If duplicate rerun ZIPs exist, move the extras elsewhere;
the notebook refuses to choose between them.


In [ ]:
SHARD_ARCHIVE_DIR_TEXT = ''  #@param {type:"string"}
SHARD_ARCHIVE_DIR = (
    Path(SHARD_ARCHIVE_DIR_TEXT) if SHARD_ARCHIVE_DIR_TEXT.strip()
    else PROJECT_DIR / 'results'
)
candidates = sorted(SHARD_ARCHIVE_DIR.glob('independent_05d_test_shard_*_of_05*.zip'))
assert candidates, f'No Stage-05D sealed shard ZIPs found in {SHARD_ARCHIVE_DIR}'
by_index = {}
for path in candidates:
    index = IA05.shard_identity(path)
    by_index.setdefault(index, []).append(path)
duplicates = {index: paths for index, paths in by_index.items() if len(paths) != 1}
assert not duplicates, (
    'Expected exactly one ZIP per shard index. Move duplicate reruns out of this folder: '
    f'{duplicates}'
)
assert sorted(by_index) == list(range(5)), f'Missing shard indices: {sorted(set(range(5)) - set(by_index))}'
SHARD_ARCHIVES_05D = [by_index[index][0] for index in range(5)]
print('Found the complete fixed shard set:')
for index, path in enumerate(SHARD_ARCHIVES_05D):
    print(f'  shard {index}: {path.name}')
print('No outcome file has been opened by this cell.')


### 3. Verify all shards, then unseal once

This cell is the formal boundary. It verifies all 625 manifested shard files before
creating the analysis directory. It then runs only the precommitted analysis. The final
directory is published atomically, and a second run is refused.


In [ ]:
OUTPUT_DIR_05E = PROJECT_DIR / 'results' / 'independent_05e_locked_analysis'
RESULT_05E = IA05.run_analysis(
    SHARD_ARCHIVES_05D,
    OUTPUT_DIR_05E,
    ANALYSIS_LOCK_TEXT,
    EXPECTED_DIGESTS['analysis'],
)
assert RESULT_05E['status'] == 'passed_one_time_locked_analysis'
assert RESULT_05E['final_literature_novelty_claim_established'] is False
print('ONE-TIME ANALYSIS STATUS:', RESULT_05E['status'])
print('RESULT ARCHIVE:', RESULT_05E['archive_path'])
print('ARCHIVE SHA-256:', RESULT_05E['archive_sha256'])


### 4. Read the confirmatory decision and calibration results

The cohort is now formally unsealed. The table below reports the exact frozen decision
components; secondary outcomes cannot rescue a failed primary gate.


In [ ]:
PRIMARY_05E = json.loads((OUTPUT_DIR_05E / 'primary_hypotheses.json').read_text())
primary_rows = []
for hypothesis, row in PRIMARY_05E['hypotheses'].items():
    primary_rows.append({
        'hypothesis': hypothesis,
        'mean_difference': row['mean_difference'],
        'ci_low': row['paired_source_bootstrap_95_ci'][0],
        'ci_high': row['paired_source_bootstrap_95_ci'][1],
        'relative_reduction': row['relative_reduction'],
        'raw_one_sided_p': row['p_value_one_sided'],
        'holm_adjusted_p': row['holm_adjusted_p_value'],
        'practical_gate': row['practical_gate_passed'],
        'statistical_gate': row['statistical_gate_passed'],
        'confirmatory_gate': row['confirmatory_gate_passed'],
    })
display(pd.DataFrame(primary_rows))
print('Both confirmatory gates passed:', PRIMARY_05E['both_confirmatory_gates_passed'])
print('Experimental novelty gate passed:', PRIMARY_05E['experimental_novelty_gate_passed'])
print('Final literature novelty claim established:', PRIMARY_05E['final_literature_novelty_claim_established'])
print(PRIMARY_05E['claim_boundary'])

CALIBRATION_05E = pd.read_csv(OUTPUT_DIR_05E / 'calibration_metrics.csv')
display(CALIBRATION_05E[CALIBRATION_05E.scope == 'all_chains'].reset_index(drop=True))


### 5. Display the locked diagnostic figures

These figures are generated entirely by the precommitted analysis: confirmatory paired
contrasts, the primary-chain risk–coverage curves, and independent reliability diagrams.


In [ ]:
for filename in (
    'primary_contrasts.png',
    'primary_risk_coverage.png',
    'independent_calibration_reliability.png',
):
    display(DisplayImage(filename=str(OUTPUT_DIR_05E / 'figures' / filename)))


## Handoff

Return the generated `independent_05e_locked_analysis_<timestamp>.zip` and this executed
notebook. Stage 05E decides the independent experimental gates. Stage 05F is the final
claim synthesis: it must combine this result with the already locked nearest-method
literature boundary and must preserve failures as carefully as successes.
